# Data Cleaning and Processing Summary

This notebook processes the raw newsletter dataset (1668 rows) through the following steps:
- fixing encoding erros and cleanswhitespace in all text columsn, the replaces common corrupted characteres
- it checks for missing values and then removes any rows missing a decription, link or title - which results in 1323 rows
- it counts fully duplicate rows (0) and row swith duplicate title-link pairs, inspects then, then removes duplicates based on title and link while keeping the first occurence leaving 1258 rows 
- find and inspect rows with dupblicate title and duplicate links, then removes the duplicates keeping the first occurence - resulting in 1192 rows
- then it fixes missing themes, counts each theme-subtheme pair
- remove unsubscribe row, normalise and clean theme text, maps themese to standard categories, then saves a dataset as "items_all_themes" (fully cleaned newsletter dataset where every item has been assigned a standardized theme using the full theme-mapping process)
- filter the datset to keep only rows wehre the new_theme is in the selected list of core themes (i.e. remove events and erp_project) 1010 rows
- created domain column for each link and used this to map each domain to an organisation and kept rows with only valid organisations leaving 903 rows (overall there are a total of 216 organisations in the dataset) 
- each organisation was mapped to a high-level sector (e.g. government, academic etc.) and a more specific sub-category within that sector: [government_public_sector: government_legislature, executive_non_departmental_public_body_ndpb, international_organisation
academic_sector: universities, academic_publisher_platform, academic_network
research_evidence_sector: research_organisation, research_institution, research_project_initiative, research_funder
civil_society_nonprofit_sector: labour_union, charity_ngo, professional_network, practitioner_organisation
knowledge_mobiliser_think_tank_sector: think_tank, evidence_mobiliser, advocacy_organisation
commercial_private_sector: consultancy, edtech_education_business, industry_association
media_sector: news_media, specialist_media, commentary_platform
digital_social_media_platforms: social_media, podcast_platform
other_miscellaneous: content_platform, cultural_organisation, government_initiative, unclear]
- created a combined 'text' columns from title and description
- saved a final cleaned dataset "items_final_themes.csv" which is the fully cleaned and processed newsletter dataset, with duplicates removed, themes standardised, organisations and categories assigned, and a combined text field ready for analysis.


In [1]:
import os 
import re 

import pandas as pd 
import numpy as np 

from ftfy import fix_text
import unicodedata as ud
from urllib.parse import urlparse

In [2]:
#load data 
df = pd.read_csv("/workspaces/AM2_erp_programme_automataion/data/interim/extracted_newsletters.csv")

In [3]:
df.head()

,id,newsletter_number,issue_date,theme,subtheme,title,description,link
0,d33c60de-7c4c-4325-8577-2a407a40d581,1,11 July 2023,DfE,NaN,"Reject fewer teacher applicants, DfE tells tra...","Susan Acland-Hood, the DfE’s permanent secreta...",https://schoolsweek.co.uk/reject-fewer-teacher...
1,c3a63fcb-9a0d-4032-bc2a-162af3377251,1,11 July 2023,DfE,NaN,Revealed: the experts advising ministers on te...,The Department for Education has appointed an ...,https://schoolsweek.co.uk/revealed-the-experts...
2,fe8c0b49-7de4-4529-b907-62d247c54e17,1,11 July 2023,Calls for evidence,NaN,Deadline 23 August 2023,Education secretary Gillian Keegan has launche...,https://schoolsweek.co.uk/chatgpt-keegan-launc...
3,f227b874-58f2-40bf-9254-7beb3f9aff8b,1,11 July 2023,Calls for evidence,NaN,ChatGPT: Keegan launches call for evidence on ...,NaN,NaN
4,8adabccc-920a-4db8-a032-58eb34b3bf24,1,11 July 2023,What are the politicians saying?,NaN,Labour,NaN,NaN


In [4]:
# Treat these text tokens as missing on read
NA_TOKENS = ["", " ", "NA", "N/A", "na", "NaN", "nan", "null", "NULL", "-"]

In [5]:
#inspect 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1924 entries, 0 to 1923
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1924 non-null   object
 1   newsletter_number  1924 non-null   int64 
 2   issue_date         1924 non-null   object
 3   theme              1924 non-null   object
 4   subtheme           114 non-null    object
 5   title              1924 non-null   object
 6   description        1543 non-null   object
 7   link               1860 non-null   object
dtypes: int64(1), object(7)
memory usage: 120.4+ KB


In [6]:
print(f"Total rows: {len(df)}")
print(f"Unique newsletter: {df['newsletter_number'].nunique()}")

Total rows: 1924
Unique newsletter: 104


# Clean Up Text

In [7]:
def clean_series(s: pd.Series) -> pd.Series:
    # Use pandas "string" dtype so NaNs stay as <NA>
    s = s.astype("string")
    mask = s.notna()
    # Fix mojibake and normalize only on non-missing cells
    s.loc[mask] = s.loc[mask].apply(fix_text)
    s.loc[mask] = s.loc[mask].apply(lambda x: ud.normalize("NFKC", x))
    # Basic whitespace cleanup
    s.loc[mask] = s.loc[mask].str.replace(r"\s+", " ", regex=True).str.strip()
    return s

# Clean all object/string columns (quick and safe)
obj_cols = [c for c in df.columns if df[c].dtype == object or pd.api.types.is_string_dtype(df[c])]
for c in obj_cols:
    df[c] = clean_series(df[c])

# Quick exact replacements for the most common artifacts (optional, simple)
REPL = {
    "Â ": " ", "Â": "",
    "‚Äì": "–", "‚Äî": "—",
    "‚Äô": "’", "‚Äò": "‘",
    "‚Äú": "“", "‚Äù": "”",
    "â€“": "–", "â€”": "—",
    "â€˜": "‘", "â€™": "’",
    "â€œ": "“", "â€\x9d": "”",
    "â€¢": "•", "â€¦": "…",
}
for c in obj_cols:
    s = df[c].astype("string")
    for bad, good in REPL.items():
        s = s.str.replace(bad, good, regex=False)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    df[c] = s

# Check for Missing Values 

In [8]:
def missing_table(d: pd.DataFrame) -> pd.DataFrame:
    mc = d.isna().sum()
    return pd.DataFrame({
        "Missing Values": mc,
        "Percentage (%)": (mc / len(d)) * 100
    }).sort_values("Missing Values", ascending=False)

print("\n=== Missing values (before drop) ===")
print(missing_table(df))


=== Missing values (before drop) ===
                   Missing Values  Percentage (%)
subtheme                     1810       94.074844
description                   381       19.802495
link                           64        3.326403
id                              0        0.000000
theme                           0        0.000000
issue_date                      0        0.000000
newsletter_number               0        0.000000
title                           0        0.000000


# Remove items where description, link or title are missing

In [9]:
# Remove rows where 'description' or 'link' is missing
df_cleaned = df.dropna(subset=['description', 'link', 'title'])

# (Optional) Check how many rows remain
print(f"Rows before: {len(df)}")
print(f"Rows after : {len(df_cleaned)}")

df = df_cleaned

Rows before: 1924
Rows after : 1521


# Check for Duplicates 

### All rows identical 

In [10]:
#All rows identical 
total_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns identical): {total_duplicates}")

Total duplicate rows (all columns identical): 0


### Title and link identical 

In [11]:
# Check duplicates where both title and link are the same
title_link_dupes = df[df.duplicated(subset=["title", "link"], keep=False)]

print(f"Number of duplicate title+link pairs: {title_link_dupes.shape[0]}")
title_link_dupes.sort_values(by=["title"]).head(2)

Number of duplicate title+link pairs: 109


,id,newsletter_number,issue_date,theme,subtheme,title,description,link
1300,906355de-4ae7-4634-8c11-aec32b6cd4ae,70,4 April 2025,Updates from the programme,<NA>,A reminder that the ESRC Education Research Pr...,"AI in Education: From chalkboards to chatbots,...",https://uk.bettshow.com/speakers/dominik-lukes
1310,9c2b5341-ecd7-470f-b968-42312c5f6c0d,71,11 April 2025,Updates from the programme,<NA>,A reminder that the ESRC Education Research Pr...,"AI in Education: From chalkboards to chatbots,...",https://uk.bettshow.com/speakers/dominik-lukes


In [12]:
title_link_dupes.theme.value_counts()

theme
Updates from the programme                                                                                                                                                                                                        35
You have indicated that you are happy to receive news and updates from the ESRC Education Research Programme. To unsubscribe, please email Elizabeth.hudson@ucl.ac.uk with the word UNSUBSCRIBE in the title of the email.        30
Updates from the Programme                                                                                                                                                                                                        10
What Matters in Education?                                                                                                                                                                                                         8
You have indicated that you are happy to receive news and updates from the ESR

In [13]:
title_link_dupes[title_link_dupes.theme == "Teacher recruitment, retention & development"]

,id,newsletter_number,issue_date,theme,subtheme,title,description,link
954,070f8e51-3829-4380-aa31-c475cb86b03b,54,22 November 2024,"Teacher recruitment, retention & development",<NA>,Education Support - Teacher Wellbeing Index,Annual report on the Teacher Wellbeing Index h...,https://www.educationsupport.org.uk/resources/...
990,6f4a2b85-478b-4b63-981c-6e127ddfc1a2,56,6 December 2024,"Teacher recruitment, retention & development",<NA>,DfE - Working lives of teachers and leaders: w...,A summary report of early findings from the th...,https://www.gov.uk/government/publications/wor...
1560,6fa09ae1-be4a-46b8-959d-bfea1be69ed5,82,11 July 2025,"Teacher recruitment, retention & development",<NA>,DfE - Working lives of teachers and leaders: w...,Findings from the third wave of the working li...,https://www.gov.uk/government/publications/wor...
1723,dd01f5d7-befb-4bf9-9f96-b3cbacadf127,92,21 November 2025,"Teacher recruitment, retention & development",<NA>,Education Support - Teacher Wellbeing Index,"For the seventh consecutive year, the annual T...",https://www.educationsupport.org.uk/resources/...


In [14]:
#drop duplicates keeping only first occurence 
df = df.drop_duplicates(subset=["title", "link"], keep="first").reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1438 entries, 0 to 1437
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1438 non-null   string
 1   newsletter_number  1438 non-null   int64 
 2   issue_date         1438 non-null   string
 3   theme              1438 non-null   string
 4   subtheme           85 non-null     string
 5   title              1438 non-null   string
 6   description        1438 non-null   string
 7   link               1438 non-null   string
dtypes: int64(1), string(7)
memory usage: 90.0 KB


### Title only duplicates

In [15]:
# Count duplicates based on title only
title_dupes = df[df.duplicated(subset=["title"], keep=False)]

print(f"Number of rows with duplicate titles: {title_dupes.shape[0]}")
title_dupes.sort_values(by="title").head(1)

Number of rows with duplicate titles: 20


,id,newsletter_number,issue_date,theme,subtheme,title,description,link
1253,8f2d0148-3cf5-4856-9a2a-a9e2839d6776,87,10 October 2025,Updates from the Programme,<NA>,Addressing key issues in teacher recruitment a...,Catch up with the video of the latest in the W...,https://mediacentral.ucl.ac.uk/Play/126585


In [16]:
title_table = title_dupes[["title", "theme"]].value_counts().reset_index(name="count")
title_table

,title,theme,count
0,Making Teaching Attractive and Worthwhile (Par...,Project news,3
1,Deadline: 28 April 2025,Political environment and key organisations,2
2,What matters in education? Education after the...,Updates from the programme,2
3,Panel:,Updates from the programme,2
4,Addressing key issues in teacher recruitment a...,Updates from the Programme,2
5,What matters in education? Education in a brok...,Updates from the programme,2
6,Labour,Political landscape & key organisations,1
7,Digital Poverty Alliance,EdTech,1
8,Digital Poverty Alliance,Thematic roundup,1
9,Panel:,"Teacher recruitment, retention & development",1


In [17]:
df = df.drop_duplicates(subset=["title"], keep="first").reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1427 entries, 0 to 1426
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1427 non-null   string
 1   newsletter_number  1427 non-null   int64 
 2   issue_date         1427 non-null   string
 3   theme              1427 non-null   string
 4   subtheme           84 non-null     string
 5   title              1427 non-null   string
 6   description        1427 non-null   string
 7   link               1427 non-null   string
dtypes: int64(1), string(7)
memory usage: 89.3 KB


### link-only duplicates 

In [18]:
# Count duplicates based on link only
link_dupes = df[df.duplicated(subset=["link"], keep=False)]

print(f"Number of rows with duplicate links: {link_dupes.shape[0]}")
link_dupes.sort_values(by="link").head(1)

Number of rows with duplicate links: 120


,id,newsletter_number,issue_date,theme,subtheme,title,description,link
1405,d5aaae06-d9db-45a8-b7a4-dc5cc274e565,102,20 February 2026,What matters in education?,<NA>,Centre for Young Lives - New report published:...,Centre for Education Systems - Comparative ana...,https://3a551fc8-7675-4cc5-9ecd-8697a47d348f.f...


In [19]:
pd.set_option("display.max_colwidth", None)

link_table = link_dupes[["link"]].value_counts().reset_index(name="count")
link_table

,link,count
0,https://www.ucl.ac.uk/education-research-programme/events/2023/oct/practical-policies-or-bright-ideas-how-particular-topics-get-front-policy-queue,4
1,https://www.ucl.ac.uk/education-research-programme/events/2024/mar/investing-early-years-priorities-and-challenges,4
2,https://childrens-participation.org/,3
3,https://uk.bettshow.com/speakers/dominik-lukes,3
4,https://www.ucl.ac.uk/education-research-programme/events/2024/jan/pupil-absence-questions-policy-research-and-practice,3
5,https://www.ucl.ac.uk/education-research-programme/events/2025/may/how-build-resilient-schools-place-based-approaches-supporting-teachers-and-leaders,3
6,https://bigeducation.org/product/next-generation-schools-conference-2024,2
7,https://3a551fc8-7675-4cc5-9ecd-8697a47d348f.filesusr.com/ugd/5d3f2a_d28e6a6811cb405aa24f207001ba1eaf.pdf,2
8,https://engagementhub.ukri.org/esrc-1/weshorizonscanningsurvey,2
9,https://epi.org.uk/events/education-priorities-for-the-next-general-election,2


In [20]:
df = df.drop_duplicates(subset=["link"], keep="first").reset_index(drop=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1363 entries, 0 to 1362
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1363 non-null   string
 1   newsletter_number  1363 non-null   int64 
 2   issue_date         1363 non-null   string
 3   theme              1363 non-null   string
 4   subtheme           80 non-null     string
 5   title              1363 non-null   string
 6   description        1363 non-null   string
 7   link               1363 non-null   string
dtypes: int64(1), string(7)
memory usage: 85.3 KB


# Identify themes and subthemes

In [21]:
#Unique counts of columns 
print("Unique titles:", df["title"].nunique())
print("Unique themes:", df["theme"].nunique())
print("Unique subthemes", df["subtheme"].nunique())
print("Unique links:", df["link"].nunique())

Unique titles: 1363
Unique themes: 72
Unique subthemes 35
Unique links: 1363


In [22]:
### Add placeholders for missing themes/subthemes
# 1) Normalize empties/whitespace/"nan"/"none" to real NA
df_norm = df.copy()
for col in ["theme", "subtheme"]:
    df_norm[col] = (
        df_norm[col]
        .astype("string")
        .replace(r"^\s*$", pd.NA, regex=True)
        .replace({"nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "none": pd.NA})
    )

# 2) Create a version that fills NA with placeholders so ALL cases are counted
df_filled = df_norm.fillna({"theme": "No theme", "subtheme": "No subtheme"})

# 3) Group and count every (theme, subtheme) combo, including placeholder cases
theme_subtheme_counts = (
    df_filled
    .groupby(["theme", "subtheme"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(by=["theme", "subtheme"])
)

# 4) Export to Excel
out_path = "/workspaces/AM2_erp_programme_automataion/data/interim/theme_subtheme_counts.xlsx"
theme_subtheme_counts.to_excel(out_path, index=False)
print(f"✅ Exported {len(theme_subtheme_counts)} rows to {out_path}")

✅ Exported 109 rows to /workspaces/AM2_erp_programme_automataion/data/interim/theme_subtheme_counts.xlsx


In [23]:
df["theme"].nunique()

72

# Check Themes and Articles 

In [24]:
# Filter articles under themes

check_themes = df[df["theme"] == "Research – Practice – Policy"].copy()

# View a few examples
display(check_themes.head(10))

,id,newsletter_number,issue_date,theme,subtheme,title,description,link
262,8da3d17e-3cba-4cb7-9300-2ae7bc665c8e,25,16 February 2024,Research – Practice – Policy,<NA>,The SHAPE of research impact,"British Academy report exploring research impact for the SHAPE disciplines, looking at the body of impact case studies submitted to the most recent research assessment exercise in the UK (REF21)",https://www.thebritishacademy.ac.uk/publications/the-shape-of-research-impact
263,a47f6842-a4d4-4ea0-b7c7-220a7cc7cf71,25,16 February 2024,Research – Practice – Policy,<NA>,NFER Event – Disadvantaged Policy webinar,Thursday 22 February 2024 – 11am Online,https://www.nfer.ac.uk/events/disadvantaged-policy-webinar
267,fbacd1f4-1928-4bf3-b104-54c495d84350,25,16 February 2024,Research – Practice – Policy,<NA>,CAPE - Quid pro quo? Why academics meet with policy professionals,"Patrick McAlary, CAPE coordinator, explores what benefits academics report from giving up their time to chat with policy professionals about their policy priorities",https://t.co/DbEx7Z1PPJ
273,bc1860dd-4c99-4899-874e-3790e719b9a9,26,23 February 2024,Research – Practice – Policy,<NA>,"BERA event - Social theory, educational research and polycrisis",22 May 2024 2pm – 4pm (Free for BERA members),https://www.bera.ac.uk/event/social-theory-educational-research-and-polycrisis-2024
278,65f60fc5-28b3-4a6e-bcb7-6c71e1784671,26,23 February 2024,Research – Practice – Policy,<NA>,Post from the Co-Production Collective - How to best engage the public to participate in collaborative projects,"Through her lived experience of working with community groups, member of the Co-Production Collective Yesmin Begum shares her key principles for meaningful co-production and involvement.",https://www.coproductioncollective.co.uk/news/how-to-best-engage-the-public-to-participate-in-collaborative-projects?dm_i=2HJW%2C1ZV0V%2C7XUV43%2C74IFU%2C1
289,6bebc104-5886-4c4c-bfe2-3d05aeaa9a06,27,1 March 2024,Research – Practice – Policy,<NA>,"DfE – Future Engagement events to facilitate collaboration between research experts, external organisations, DfE analysts and policy colleagues on each of the DfE Areas of Research Interest (ARI).","If you would like to stay up to date with these events, please get in touch with the DfE research engagement team so it can ensure you get an invitation.",mailto:research.engagement@education.gov.uk
290,72e736b1-e2a2-43a9-ac5f-f96189478a33,27,1 March 2024,Research – Practice – Policy,<NA>,Event – The impact of Impact on evidence-informed education: celebrating 20 issues,"7th March, 16:30 - 17:30 Online panel discussion In this panel discussion, Impact are bringing back past guest editors to discuss how they think Impact has impacted evidence-informed education since its first publication, where it sits within the evidence-informed education ecosystem and where they see its future role.",https://my.chartered.college/event/the-impact-of-impact-on-evidence-informed-education-celebrating-20-issues?dm_i=449V%2C1LWBC%2C8F3NAU%2C7JLNJ%2C1
291,aa279d6e-4615-4f6a-a13a-5bfbb12b3155,27,1 March 2024,Research – Practice – Policy,<NA>,Schools Week - Teacher-powered research: how we're building the evidence around everyday practice,A new stream of EEF evaluations aims to give teachers better evidence to support their everyday practice Read the article,https://schoolsweek.co.uk/teacher-powered-research-how-were-building-the-evidence-around-everyday-practice
292,22ed37ae-6a5d-4476-8a20-5a06e2ae8917,27,1 March 2024,Research – Practice – Policy,<NA>,Nuffield - Provocations: what are the issues shaping tomorrow?,Nuffield want to fund research that can address tomorrow's challenges and influence policy to prioritise social well-being. Click on the link to read the agenda setting provocations from leading thinkers in the field,https://www.nuffieldfoundation.org/news/opinion/provocations-issues-shaping-tomorrow
303,f09c6509-114c-4d4c-b630-d5d60c014152,28,8 March 2024,Research – Practice – Policy,<NA>

In [25]:
df[df["theme"] == "4 Nations & key organisations"][["theme", "subtheme", "title"]].head(10)

,theme,subtheme,title
453,4 Nations & key organisations,<NA>,NASUWT comments on Education (Scotland) Bill
458,4 Nations & key organisations,<NA>,"Belfast Telegraph - Third of children in NI have too few places to play, survey reveals"
467,4 Nations & key organisations,<NA>,Politics and Policy blog - Understanding evidence in policy-making
468,4 Nations & key organisations,<NA>,Scottish Government - New approaches to help eradicate child poverty


In [26]:
df[df["theme"].isin(["Update from the projects", "Updates from the projects", 
                      "Update from the programme", "Updates from the programme",
                      "Updates from the Programme"])][["theme", "subtheme", "title"]].head(20)

,theme,subtheme,title
250,Update from the projects,From Equity in EdTech,"Unpicking a school's technology ""culture"""
415,Updates from the projects,ERP project: Investigating the recruitment and retention of ethnic minority teachers PI: Stephen Gorard,The Conversation - New faith schools in England could soon allocate all their places on religious grounds – here's why that's a bad idea
441,Updates from the projects,Embedding children's participation rights in pedagogical practice in lower primary classrooms in Wales PI: Sarah Chicken,BERA Blog - Can the Curriculum for Wales and 'cynefin' enable children's participative rights in schools?
442,Updates from the projects,Embedding children's participation rights in pedagogical practice in lower primary classrooms in Wales PI: Sarah Chicken,"Part of a BERA blog special issues - Doing cynefin: Exploring ideas on belonging, connectedness and community in the Curriculum for Wales"
448,Updates from the projects,Sustainable school leadership: comparing approaches to the training supply and retention of senior school leaders across the UK PI: Toby Greany,The Conversation - School suspensions are on the rise – secondary schools can tackle this by creating a sense of belonging
454,Updates from the programme,<NA>,What matters in education? Education after the election: priorities for change
455,Updates from the programme,<NA>,Catch up with the latest briefing notes from 'What matters in education?'
460,Updates from the projects,Investigating the recruitment and retention of ethnic minority teachers PI: Stephen Gorard,The Conversation - Why UK government policies have failed to recruit enough teachers for years
480,Updates from the programme,<NA>,Join us for our next event: What matters in education? More or less technology in the classroom – the value and purposes of technology use in school
490,Updates from the projects,<NA>,Steph Ainsworth shared some of the emerging findings from 'Decentring the resilient teacher: exploring interactions between individuals and their social ecologies' with Alistair Bryce-Clegg as part of the TTS Talking Early Years podcast series.


In [27]:
df[df["theme"].str.contains("Thematic", case=False, na=False)][["theme", "subtheme", "title"]].head(20)

,theme,subtheme,title
5,Thematic roundup,Digital,Digital Poverty Alliance
6,Thematic roundup,Digital,A long read from Nuffield funded research project ' Advancing Leadership Development in Early Years Education via Digitally Mediated Professional Learning'
9,Thematic roundup,"Teacher recruitment, retention and development",Teacher retention commission: 8 proposals to stem exodus
10,Thematic roundup,"Teacher recruitment, retention and development",Who's supporting school leaders to stop them hitting crisis point?
19,Thematic roundup,Digital,Tony Blair Institute - The Future of Learning: Delivering Tech-Enabled Quality Education for Britain
20,Thematic roundup,"Teacher recruitment, retention and development",NFER Research - Policy options for a long-term teacher pay and financial incentives strategy
21,Thematic roundup,"Teacher recruitment, retention and development","Consider 'golden handcuff' bursaries, major ITT providers say"
28,Thematic roundup,Digital,Nesta - Is edtech for under-fives about to boom?
30,Thematic roundup,"Teacher recruitment, retention & development",'Being well' and 'doing well': Exploring the relationship between student and teacher wellbeing
31,Thematic roundup,"Teacher recruitment, retention & development",Re pay awards: School Teachers' Review Body 33rd report: 2023


# Rename Themes

In [28]:
# ---------- 0) Drop rows where the entire theme is the unsubscribe text
UNSUB_THEME = (
    "You have indicated that you are happy to receive news and updates from the "
    "ESRC Education Research Programme. To unsubscribe, please email "
    "Elizabeth.hudson@ucl.ac.uk with the word UNSUBSCRIBE in the title of the email."
)
mask_unsub = df["theme"].astype(str).str.strip().eq(UNSUB_THEME)
dropped_rows = int(mask_unsub.sum())
df = df[~mask_unsub].copy()

# ---------- 1) Normalizers
def norm_theme(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    s = s.replace("—", "-").replace("–", "-")
    s = s.replace("\u2018", "'").replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    return s.lower()

def norm_key(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = s.replace("—", " ").replace("–", " ").replace("-", " ")
    s = s.replace("&", " and ")
    s = s.replace("\u2018", "'").replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
    s = re.sub(r"[,\.\u00A0]", " ", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ---------- 2) Theme mapping
pairs = [
    # ============ update_from_programme ============
    ("update_from_programme", "Programme Update"),
    ("update_from_programme", "Programme news"),
    ("update_from_programme", "Programme update"),
    ("update_from_programme", "Updates from the Programme"),
    ("update_from_programme", "Updates from the programme"),
    ("update_from_programme", "Update from the ESRC Education Research Programme"),
    ("update_from_programme", "Updates from the ESRC"),
    ("update_from_programme", "Conferences"),
    ("update_from_programme", "Events"),
    ("update_from_programme", "Opportunities"),
    ("update_from_programme", "Opportunities for funding"),
    ("update_from_programme", "Opportunities to blog"),
    ("update_from_programme", "Relevant Events"),
    ("update_from_programme", "Seminar series topics"),
    ("update_from_programme", "Seminar topics"),

    # ============ update_from_pi ============
    ("update_from_pi", "PI Updates and Papers"),
    ("update_from_pi", "PI: David Lundie"),
    ("update_from_pi", "Toby Greany"),
    ("update_from_pi", "News from the Projects"),
    ("update_from_pi", "News from the projects"),
    ("update_from_pi", "Project news"),
    ("update_from_pi", "Update from the ERP projects"),
    ("update_from_pi", "Update from the projects"),
    ("update_from_pi", "Updates from David Lundie"),
    ("update_from_pi", "Updates from Steph Ainsworth"),
    ("update_from_pi", "Updates from the ERP projects"),
    ("update_from_pi", "Updates from the projects"),
    ("update_from_pi", "Embedding children's participation rights in pedagogical practice in lower primary classrooms in Wales PI: Sarah Chicken"),
    ("update_from_pi", "Investigating the recruitment and retention of ethnic minority teachers PI: Stephen Gorard"),
    ("update_from_pi", "Rethinking teacher recruitment: New approaches to attracting prospective STEM teachers PI: Rob Klassen"),
    ("update_from_pi", "Sustainable school leadership: comparing approaches to the training, supply and retention of senior school leaders across the UK PI Toby Greany"),
    ("update_from_pi", "Towards equity focused approaches to EdTech: a socio-technical perspective PI: Professor Rebecca Eynon"),
    ("update_from_pi", "Towards equity focused approaches to EdTech: a socio-technical perspective PI: Rebecca Eynon"),
    ("update_from_pi", "Decentring the 'resilient teacher': exploring interactions between individuals and their social ecologies PI: Steph Ainsworth"),
    ("update_from_pi", "Peer reviewed articles from the ERP projects"),
    ("update_from_pi", "Peer reviewed publications from the ERP projects"),
    ("update_from_pi", "Updates from Stephen Gorard"),
    ("update_from_pi", "Updates from Toby Greany"),
    ("update_from_pi", "Updates from Sarah Chicken"),
    ("update_from_pi", "Updates from PI Steven Gorard"),
    ("update_from_pi", "Update from PI Stephanie Ainsworth"),
    ("update_from_pi", "Updates from PI Sarah Chicken"),
    ("update_from_pi", "Update from PI Robert Klassen"),
    ("update_from_pi", "Update from PI David Lundie"),
    ("update_from_pi", "Update from PI Alison Porter"),

    # ============ what_matters_ed ============
    ("what_matters_ed", "What Matters in Education?"),
    ("what_matters_ed", "What matters in education?"),

    # ============ teacher_rrd ============
    ("teacher_rrd", "Teacher recruitment, retention & development"),

    # ============ edtech ============
    ("edtech", "EdTech"),

    # ============ four_nations ============
    ("four_nations", "4 Nations"),
    ("four_nations", "4 Nations & key organisations"),
    ("four_nations", "Four Nations"),
    ("four_nations", "Four Nations Landscape"),
    ("four_nations", "Four Nations landscape"),
    ("four_nations", "Four nations"),
    ("four_nations", "Political landscape across Four Nations & key organisations"),

    # ============ policy_practice_research ============
    ("policy_practice_research", "Research \u2013 Practice \u2013 Policy"),
    ("policy_practice_research", "Education, Policy & Practice"),
    ("policy_practice_research", "Calls for evidence"),
    ("policy_practice_research", "Other Reports"),
    ("policy_practice_research", "Other Research"),
    ("policy_practice_research", "Relevant Research"),
    ("policy_practice_research", "Reports"),
    ("policy_practice_research", "Research"),

    # ============ political_environment_key_organisations ============
    ("political_environment_key_organisations", "What are the politicians saying?"),
    ("political_environment_key_organisations", "Political environment and key organisations"),
    ("political_environment_key_organisations", "Political landscape - the election"),
    ("political_environment_key_organisations", "Political landscape & key organisations"),
    ("political_environment_key_organisations", "DfE"),
    ("political_environment_key_organisations", "EEF"),
    ("political_environment_key_organisations", "ESRC"),
    ("political_environment_key_organisations", "Politics"),
    ("political_environment_key_organisations", "Launch of ESRC survey on social science research skills"),
    ("political_environment_key_organisations", "Updates from UKRI"),
    ("political_environment_key_organisations", "Update from UKRI"),
]

# ---------- 3) Build lookup
lookup = {norm_theme(curr): new for new, curr in pairs}

# ---------- 4) Apply theme mapping
theme_norm = df["theme"].map(norm_theme)
df["new_theme"] = theme_norm.map(lookup)

# ---------- 4b) Keyword overrides
kw_four_nations = theme_norm.str.contains(r"\b(4|four) nations\b", regex=True, na=False)
df.loc[kw_four_nations, "new_theme"] = "four_nations"

# ---------- 5) Thematic roundup — split by subtheme, then other overrides
sub_norm = df["subtheme"].map(norm_key)

kw_thematic = theme_norm.str.contains(r"thematic\s+round", regex=True, na=False)
df.loc[kw_thematic & sub_norm.str.contains("digital", na=False), "new_theme"] = "edtech"
df.loc[kw_thematic & sub_norm.str.contains("teacher", na=False), "new_theme"] = "teacher_rrd"
df.loc[kw_thematic & df["new_theme"].isna(), "new_theme"] = "update_from_programme"

target_rrd = "teacher recruitment retention and development"
df.loc[sub_norm.eq(target_rrd), "new_theme"] = "teacher_rrd"
df.loc[sub_norm.eq("digital"), "new_theme"] = "edtech"

# ---------- 6) Fill remaining unmapped
df["new_theme"] = df["new_theme"].fillna(df["theme"])

# ---------- 7) Export
summary = (
    df.assign(theme_norm=theme_norm, subtheme_norm=sub_norm)
      .groupby(["new_theme", "theme_norm"], dropna=False)
      .size()
      .reset_index(name="count")
      .sort_values(["new_theme", "count"], ascending=[True, False])
)

summary_path = "/workspaces/AM2_erp_programme_automataion/data/interim/theme_mapping_summary.xlsx"

with pd.ExcelWriter(summary_path) as xw:
    df.to_excel(xw, sheet_name="data_with_new_theme", index=False)
    summary.to_excel(xw, sheet_name="mapping_summary", index=False)

print(f"✅ Dropped {dropped_rows} unsubscribe row(s).")
print("✅ Mapping applied.")
print("📄 Excel written to:", summary_path)

unmapped = df[~df["new_theme"].isin([
    "update_from_programme", "update_from_pi", "what_matters_ed",
    "teacher_rrd", "edtech", "four_nations",
    "policy_practice_research", "political_environment_key_organisations"
])]["new_theme"].value_counts()
print("\n⚠️ Unmapped themes:\n", unmapped)

/tmp/ipykernel_26939/38658076.py:135: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kw_four_nations = theme_norm.str.contains(r"\b(4|four) nations\b", regex=True, na=False)


✅ Dropped 0 unsubscribe row(s).
✅ Mapping applied.
📄 Excel written to: /workspaces/AM2_erp_programme_automataion/data/interim/theme_mapping_summary.xlsx

⚠️ Unmapped themes:
 Series([], Name: count, dtype: int64)


In [29]:
# ---------- 7) View unique new_theme values and their counts
theme_counts = (
    df["new_theme"]
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={"index": "new_theme", "new_theme": "count"})
)

print("🧭 Unique new_theme values and their counts:")
print(theme_counts)

🧭 Unique new_theme values and their counts:
                                     count  count
0  political_environment_key_organisations    258
1                          what_matters_ed    209
2                              teacher_rrd    203
3                                   edtech    193
4                 policy_practice_research    192
5                             four_nations    131
6                           update_from_pi    103
7                    update_from_programme     74


In [30]:
all_clean_path = "/workspaces/AM2_erp_programme_automataion/data/preprocessed/newsletters_clean.csv"
df.to_csv(all_clean_path, index=False)

# Number of unique domain names 

In [31]:
# Extract domain names from the 'link' column
df["domain"] = df["link"].apply(lambda x: urlparse(str(x)).netloc if pd.notna(x) else None)

# Count unique domains
unique_domains = df["domain"].nunique()

print(f"🌐 There are {unique_domains} unique domains in this dataset.")

# Optional: see the top 10 most common domains
domain_counts = df["domain"].value_counts().reset_index()
domain_counts.columns = ["domain", "count"]
print(domain_counts.head(60))

🌐 There are 381 unique domains in this dataset.
                                   domain  count
0                       schoolsweek.co.uk    160
1                              www.gov.uk     71
2                           www.ucl.ac.uk     41
3                     theconversation.com     35
4                          www.bera.ac.uk     30
5                     www.theguardian.com     30
6                              epi.org.uk     29
7                          www.nfer.ac.uk     25
8                    www.eventbrite.co.uk     23
9                 www.education-ni.gov.uk     22
10                           www.gov.scot     20
11  bera-journals.onlinelibrary.wiley.com     19
12               committees.parliament.uk     18
13             www.belfasttelegraph.co.uk     17
14                            www.tes.com     17
15             ffteducationdatalab.org.uk     15
16                          www.gov.wales     14
17             www.nuffieldfoundation.org     13
18      www.institute

In [32]:
review_path = "/workspaces/AM2_erp_programme_automataion/data/interim/domain_review.xlsx"
domain_counts.to_excel(review_path, index=False)
print("Wrote:", review_path)

Wrote: /workspaces/AM2_erp_programme_automataion/data/interim/domain_review.xlsx


# Domain to Organisations

In [33]:
df.head(0)

,id,newsletter_number,issue_date,theme,subtheme,title,description,link,new_theme,domain


In [34]:
domain_to_org = {
    # schools_week
    "schoolsweek.co.uk": "schools_week",

    # uk_government
    "www.gov.uk": "uk_government",
    "assets.publishing.service.gov.uk": "uk_government",
    "educationhub.blog.gov.uk": "uk_government",
    "publicpolicydesign.blog.gov.uk": "uk_government",
    "openpolicy.blog.gov.uk": "uk_government",
    "explore-education-statistics.service.gov.uk": "uk_government",
    "consult.education.gov.uk": "uk_government",
    "teaching-vacancies.service.gov.uk": "uk_government",
    "deprivation.communities.gov.uk": "uk_government",
    "fellows.ai.gov.uk": "uk_government",

    # scottish_government
    "www.gov.scot": "scottish_government",
    "blogs.gov.scot": "scottish_government",

    # welsh_government
    "www.gov.wales": "welsh_government",
    "educationwales.blog.gov.wales": "welsh_government",
    "hwb.gov.wales": "welsh_government",
    "www.medr.cymru": "welsh_government",

    # ni_government
    "www.education-ni.gov.uk": "ni_government",
    "www.economy-ni.gov.uk": "ni_government",
    "www.health-ni.gov.uk": "ni_government",

    # uk_parliament
    "committees.parliament.uk": "uk_parliament",
    "commonslibrary.parliament.uk": "uk_parliament",
    "hansard.parliament.uk": "uk_parliament",
    "lordslibrary.parliament.uk": "uk_parliament",
    "post.parliament.uk": "uk_parliament",
    "publications.parliament.uk": "uk_parliament",
    "researchbriefings.files.parliament.uk": "uk_parliament",
    "www.parliament.uk": "uk_parliament",
    "whatson.parliament.uk": "uk_parliament",
    "parliamentlive.tv": "uk_parliament",

    # welsh_parliament
    "senedd.wales": "welsh_parliament",
    "business.senedd.wales": "welsh_parliament",
    "research.senedd.wales": "welsh_parliament",

    # scottish_parliament
    "www.parliament.scot": "scottish_parliament",

    # education_scotland
    "education.gov.scot": "education_scotland",

    # ofsted
    "educationinspection.blog.gov.uk": "ofsted_blog",

    # oecd
    "www.oecd.org": "oecd",
    "www.oecd-ilibrary.org": "oecd",
    "newsletter.oecd.org": "oecd",
    "www.oecd-events.org": "oecd",
    "oecdedutoday.com": "oecd",
    "one.oecd.org": "oecd",

    # ucl
    "www.ucl.ac.uk": "ucl",
    "blogs.ucl.ac.uk": "ucl",
    "mediacentral.ucl.ac.uk": "ucl",
    "discovery.ucl.ac.uk": "ucl",
    "profiles.ucl.ac.uk": "ucl",
    "uclpress.scienceopen.com": "ucl",
    "doi-org.libproxy.ucl.ac.uk": "ucl",
    "www-nature-com.libproxy.ucl.ac.uk": "ucl",
    "ucl.us20.list-manage.com": "ucl",

    # uwe_bristol
    "blogs.uwe.ac.uk": "uwe_bristol_blog",
    "www.uwe.ac.uk": "uwe_bristol_blog",

    # bera
    "www.bera.ac.uk": "bera",
    "bera.us9.list-manage.com": "bera",

    # bera_journals
    "bera-journals.onlinelibrary.wiley.com": "bera_journals",

    # conversation
    "theconversation.com": "conversation",

    # guardian
    "www.theguardian.com": "guardian",
    "observer.co.uk": "guardian",

    # epi
    "epi.org.uk": "epi",
    "contacts.epi.org.uk": "epi",
    "epi.us15.list-manage.com": "epi",

    # nfer
    "www.nfer.ac.uk": "nfer",
    "nfer.ac.uk": "nfer",

    # fft_ed_datalab
    "ffteducationdatalab.org.uk": "fft_ed_datalab",
    "ffteducationdatalab.us12.list-manage.com": "fft_ed_datalab",

    # nuffield
    "www.nuffieldfoundation.org": "nuffield",
    "nuffieldfoundation.cmail19.com": "nuffield",
    "nuffieldfoundation.cmail20.com": "nuffield",

    # ifg
    "www.instituteforgovernment.org.uk": "ifg",

    # upen
    "upen.ac.uk": "upen",
    "www.upen.ac.uk": "upen",
    "upen.us14.list-manage.com": "upen",

    # university_of_birmingham
    "blog.bham.ac.uk": "university_of_birmingham",
    "www.birmingham.ac.uk": "university_of_birmingham",

    # ifs
    "ifs.org.uk": "ifs",

    # fed
    "fed.education": "fed",

    # bbc
    "www.bbc.co.uk": "bbc",
    "bbc.co.uk": "bbc",

    # tes
    "www.tes.com": "tes",

    # eef
    "educationendowmentfoundation.org.uk": "eef",

    # chartered_college_of_teaching
    "my.chartered.college": "chartered_college_of_teaching",
    "chartered.college": "chartered_college_of_teaching",
    "news.chartered.college": "chartered_college_of_teaching",

    # childrens_commissioner
    "www.childrenscommissioner.gov.uk": "childrens_commissioner",

    # british_academy
    "www.thebritishacademy.ac.uk": "british_academy",
    "email.thebritishacademy.ac.uk": "british_academy",
    "thebritishacademyecrn.com": "british_academy",

    # teacher_tapp
    "teachertapp.co.uk": "teacher_tapp",
    "teachertapp.com": "teacher_tapp",

    # ippo
    "theippo.co.uk": "ippo",

    # hepi
    "www.hepi.ac.uk": "hepi",

    # ippr
    "www.ippr.org": "ippr",
    "ippr-org.files.svdcdn.com": "ippr",

    # joseph_rowntree_foundation
    "www.jrf.org.uk": "joseph_rowntree_foundation",

    # nesta
    "www.nesta.org.uk": "nesta",

    # national_audit_office
    "www.nao.org.uk": "national_audit_office",
    "news.comms.nao.org.uk": "national_audit_office",

    # belfast_telegraph
    "www.belfasttelegraph.co.uk": "belfast_telegraph",

    # independent
    "www.independent.co.uk": "independent",

    # inews
    "inews.co.uk": "inews",
    "link.news.inews.co.uk": "inews",

    # wonkhe
    "wonkhe.com": "wonkhe",
    "wonkhe.cmail20.com": "wonkhe",

    # fe_week
    "feweek.co.uk": "fe_week",

    # national_literacy_trust
    "literacytrust.org.uk": "national_literacy_trust",

    # national_education_union
    "neu.org.uk": "national_education_union",

    # 5rights_foundation
    "5rightsfoundation.com": "5rights_foundation",

    # linkedin
    "www.linkedin.com": "linkedin",

    # twitter / x
    "twitter.com": "twitter",
    "x.com": "twitter",

    # ukri
    "www.ukri.org": "ukri",
    "gtr.ukri.org": "ukri",
    "engagementhub.ukri.org": "ukri",

    # ada_lovelace_institute
    "www.adalovelaceinstitute.org": "ada_lovelace_institute",

    # sutton_trust
    "www.suttontrust.com": "sutton_trust",

    # ascl
    "www.ascl.org.uk": "ascl",
    "ascl.org.uk": "ascl",

    # digital_poverty_alliance
    "digitalpovertyalliance.org": "digital_poverty_alliance",

    # edge_foundation
    "www.edge.co.uk": "edge_foundation",

    # centre_for_education_and_youth
    "cfey.org": "centre_for_education_and_youth",

    # society_for_research_into_higher_education
    "srhe.ac.uk": "society_for_research_into_higher_education",

    # harvard
    "www.gse.harvard.edu": "harvard_graduate_school_of_education",
    "click.communications.gse.harvard.edu": "harvard_graduate_school_of_education",
    "scholar.harvard.edu": "harvard_graduate_school_of_education",

    # transforming_society
    "www.transformingsociety.co.uk": "transforming_society",

    # researchgate
    "www.researchgate.net": "researchgate",

    # lse
    "www.lse.ac.uk": "london_school_of_economics",
    "eprints.lse.ac.uk": "london_school_of_economics",
    "cep.lse.ac.uk": "centre_for_economic_performance_lse",

    # tony_blair_institute
    "www.institute.global": "tony_blair_institute",
    "institute.global": "tony_blair_institute",

    # cpag
    "cpag.org.uk": "child_poverty_action_group",

    # crae
    "crae.org.uk": "children_rights_alliance_england",

    # demos
    "demos.co.uk": "demos",

    # wales_centre_for_public_policy
    "wcpp.org.uk": "wales_centre_for_public_policy",
    "wcpp.us12.list-manage.com": "wales_centre_for_public_policy",

    # politics_home
    "www.politicshome.com": "politics_home",

    # transforming_evidence
    "transforming-evidence.org": "transforming_evidence",

    # nursery_world
    "www.nurseryworld.co.uk": "nursery_world_magazine",

    # naht
    "www.naht.org.uk": "national_association_head_teachers",

    # nasuwt
    "www.nasuwt.org.uk": "nasuwt_teachers_union",

    # ripl
    "ripl.uk": "research_improvement_for_policy_and_learning",

    # internet_matters
    "www.internetmatters.org": "internet_matters",

    # cape
    "www.cape.ac.uk": "cape_collaboration_for_public_engagement",

    # coproduction_collective
    "www.coproductioncollective.co.uk": "coproduction_collective",

    # options2040
    "options2040.co.uk": "options_2040_project",

    # alan_turing_institute
    "www.turing.ac.uk": "alan_turing_institute",

    # unicef
    "www.unicef.org": "unicef",

    # n8
    "www.n8research.org.uk": "n8_research_partnership",

    # charities_supporting_teachers
    "cstuk.org.uk": "charities_supporting_teachers_uk",

    # taylor_and_francis
    "www.tandfonline.com": "taylor_and_francis",
    "insights.taylorandfrancis.com": "taylor_and_francis",

    # sage_journals
    "journals.sagepub.com": "sage_journals",

    # oii_edtech
    "edtech.oii.ox.ac.uk": "oii_edtech_equity",

    # childrens_participation
    "childrens-participation.org": "childrend_participation_in_schools",

    # fairness_foundation
    "fairnessfoundation.com": "fairness_foundation",
    "files.fairnessfoundation.com": "fairness_foundation",

    # youth_endowment_fund
    "youthendowmentfund.org.uk": "youth_endowment_fund",

    # teach_first
    "www.teachfirst.org.uk": "teach_first",

    # national_institute_of_teaching
    "niot.org.uk": "national_institute_of_teaching",
    "niot.s3.amazonaws.com": "national_institute_of_teaching",

    # oriel_square
    "www.orielsquare.co.uk": "oriel_square",

    # daily_mirror
    "www.mirror.co.uk": "daily_mirror",

    # daily_telegraph
    "www.telegraph.co.uk": "daily_telegraph",

    # full_fact
    "fullfact.org": "full_fact",

    # fe_news
    "www.fenews.co.uk": "fe_news",

    # pearson
    "www.pearson.com": "pearson",

    # scottish_ai
    "www.scottishai.com": "scottish_ai",

    # universitas_21
    "universitas21.com": "universitas_21",

    # unesco
    "www.unesco.org": "unesco",
    "unesdoc.unesco.org": "unesco",

    # education_support_charity
    "www.educationsupport.org.uk": "education_support_charity",

    # teaching_commission
    "teachingcommission.co.uk": "teaching_commission",

    # centre_for_young_lives
    "www.centreforyounglives.org.uk": "centre_for_young_lives",

    # wiley
    "onlinelibrary.wiley.com": "wiley",
    "el.wiley.com": "wiley",

    # elsevier
    "www.elsevier.com": "elsevier",
    "www.sciencedirect.com": "sciencedirect",
    "researcheracademy.elsevier.com": "elsevier_researcher_academy",

    # mdpi
    "www.mdpi.com": "mdpi_journals",

    # frontiers
    "www.frontiersin.org": "frontiers_journal",

    # repec
    "ideas.repec.org": "repec_ideas",
    "econpapers.repec.org": "repec_econpapers",

    # jstor
    "www.jstor.org": "jstor",
    "daily.jstor.org": "jstor_daily",

    # education_arxiv
    "edarxiv.org": "education_arxiv",

    # doi
    "doi.org": "doi_via_ucl_proxy",

    # leverhulme
    "www.leverhulme.ac.uk": "leverhulme_trust",

    # substacks / commentary
    "profbeckyallen.substack.com": "becky_allen_substack",
    "rebeccaallen.co.uk": "rebecca_allen",
    "magicsmoke.substack.com": "magicsmoke_substack",
    "samf.substack.com": "samf_substack",
    "benniekara.substack.com": "bennie_kara_substack",
    "blog.policy.manchester.ac.uk": "policy_manchester_blog",

    # academy_of_social_sciences
    "acss.org.uk": "academy_of_social_sciences",
    "acss.civiplus.net": "academy_of_social_sciences",

    # national_education_policy_center
    "nepc.colorado.edu": "national_education_policy_center",

    # echild
    "www.echild.ac.uk": "echild_research_centre",

    # ari
    "ari.org.uk": "ari_association_for_research_innovation",

    # working_lives_of_teachers
    "www.workinglivesofteachers.com": "working_lives_of_teachers",

    # shadow_panel
    "shadowpanel.uk": "shadow_panel_project",

    # sky_news
    "news.sky.com": "sky_news",

    # the_times
    "www.thetimes.com": "the_times",

    # times_literary_supplement
    "www.the-tls.co.uk": "times_literary_supplement",

    # yorkshire_post
    "www.yorkshirepost.co.uk": "yorkshire_post",

    # financial_times
    "www.ft.com": "financial_times",

    # hechinger_report
    "hechingerreport.org": "hechinger_report",

    # holyrood_magazine
    "www.holyrood.com": "holyrood_magazine",

    # nation_cymru
    "nation.cymru": "nation_cymru",

    # evening_standard
    "www.standard.co.uk": "evening_standard",

    # fda_union
    "www.fda.org.uk": "fda_union",

    # sustainable_school_leadership
    "sustainableschoolleadership.uk": "sustainable_school_leadership",

    # big_education
    "bigeducation.org": "big_education",

    # edsk
    "edsk.org": "edsk_think_tank",

    # bett
    "uk.bettshow.com": "bett_show",

    # tech_uk
    "www.techuk.org": "tech_uk",

    # edtech_digest
    "www.edtechdigest.com": "edtech_digest",

    # techbullion
    "techbullion.com": "techbullion",

    # wired_gov
    "www.wired-gov.net": "wired_gov",

    # labour
    "labourlist.org": "labour_list",
    "labour.org.uk": "labour_party",

    # labour_together
    "www.labourtogether.uk": "labour_together",

    # liberal_democrats
    "www.libdems.org.uk": "liberal_democrats",

    # on_think_tanks
    "onthinktanks.org": "on_think_tanks",

    # chandler_institute
    "www.chandlerinstitute.org": "chandler_institute",

    # issuu
    "issuu.com": "issuu",

    # e_estonia
    "e-estonia.com": "e_estonia",

    # office_for_national_statistics
    "www.ons.gov.uk": "office_for_national_statistics",

    # barnardos
    "www.barnardos.org.uk": "barnardos",

    # magic_breakfast
    "www.magicbreakfast.com": "magic_breakfast",

    # nspcc
    "learning.nspcc.org.uk": "nspcc_learning",

    # centre_for_social_justice
    "www.centreforsocialjustice.org.uk": "centre_for_social_justice",

    # centre_for_mental_health
    "www.centreformentalhealth.org.uk": "centre_for_mental_health",

    # action_for_children
    "media.actionforchildren.org.uk": "action_for_children",

    # play_wales
    "play.wales": "play_wales",

    # children_in_wales
    "www.childreninwales.org.uk": "children_in_wales",

    # queen_mary_university_london
    "www.qmul.ac.uk": "queen_mary_university_london",

    # kings_college_london
    "www.kcl.ac.uk": "kings_college_london",

    # kings_fund
    "kingsfundmail.org.uk": "kings_fund",

    # nottingham_trent_university
    "www.ntu.ac.uk": "nottingham_trent_university",

    # university_of_dundee
    "dundee.onlinesurveys.ac.uk": "university_of_dundee_surveys",

    # university_of_east_anglia
    "etat.uea.ac.uk": "university_of_east_anglia",

    # manchester_metropolitan_university
    "www.mmu.ac.uk": "manchester_metropolitan_university",

    # university_of_bristol
    "www.bristol.ac.uk": "university_of_bristol",

    # university_of_nottingham
    "www.nottingham.ac.uk": "university_of_nottingham",

    # early_years_alliance
    "www.eyalliance.org.uk": "early_years_alliance",

    # ambition_institute
    "www.ambition.org.uk": "ambition_institute",

    # twinkl
    "www.twinkl.co.uk": "twinkl",

    # ocr
    "www.ocr.org.uk": "ocr_exam_board",

    # digital_good_network
    "digitalgood.net": "digital_good_network",

    # edtech_innovation_hub
    "www.edtechinnovationhub.com": "edtech_innovation_hub",

    # edtech_strategy_lab
    "www.edtechstrategylab.org": "edtech_strategy_lab",

    # atkins_realis
    "www.atkinsrealis.com": "atkins_realis",

    # beyth_consultancy
    "beyth.co.uk": "beyth_consultancy",

    # staff_college
    "thestaffcollege.uk": "staff_college",

    # association_of_directors_of_childrens_services
    "adcs.org.uk": "association_of_directors_of_childrens_services",

    # headmasters_and_headmistresses_conference
    "www.hmc.org.uk": "headmasters_and_headmistresses_conference",

    # national_crime_agency
    "www.nationalcrimeagency.gov.uk": "national_crime_agency",

    # dods_monitoring
    "downloads2.dodsmonitoring.com": "dods_monitoring",
    "mmail.dods.co.uk": "dods_monitoring",

    # local_government_information_unit
    "lgiu.org": "local_government_information_unit",

    # local_government_chronicle
    "www.lgcplus.com": "local_government_chronicle",

    # gamayo
    "gamayo.co.uk": "gamayo",

    # dutch_government
    "www.rijksoverheid.nl": "dutch_government",

    # education_appg
    "educationappg.org.uk": "education_appg",
    "www.schoolsappg.org.uk": "all_party_parliamentary_group_schools",

    # cipr
    "cipr.co.uk": "chartered_institute_of_public_relations",

    # upp_foundation
    "upp-foundation.org": "upp_foundation",

    # research_on_research_institute
    "researchonresearch.org": "research_on_research_institute",

    # arc_institute
    "arcinstitute.org": "arc_institute",

    # oxford_school_of_thought
    "www.oxfordschoolofthought.org": "oxford_school_of_thought",

    # northern_health_science_alliance
    "www.thenhsa.co.uk": "northern_health_science_alliance",

    # council_of_europe
    "www.coe.int": "council_of_europe",

    # cambridge_university_press
    "www.cambridge.org": "cambridge_university_press",

    # uk_civil_service_jobs
    "www.civilservicejobs.service.gov.uk": "uk_civil_service_jobs",

    # london_design_biennale
    "londondesignbiennale.com": "london_design_biennale",

    # unclear/small
    "the-difference.com": "the_difference",
    "www.the-difference.com": "the_difference",
    "teachersuccess.co.uk": "teacher_success",
    "inclusioninpractice.org.uk": "inclusion_in_practice",
    "www.innovate-ed.uk": "innovate_ed",
    "www.insideedgetraining.co.uk": "inside_edge_training",
    "www.funding-futures.org": "funding_futures",
    "digitalyouthindex.uk": "digital_youth_index",
    "newvisionsforeducation.org.uk": "new_visions_for_education",
    "tpea.ac.uk": "tpea_association",
    "localed2025.org.uk": "local_ed_2025",
    "lucaf.org": "lucas_education_foundation",
}

# Apply — anything not in dict becomes NaN, no rows dropped
df["organisation"] = df["domain"].map(domain_to_org)

# Sanity check
missing = df[df["organisation"].isna()]["domain"].value_counts()
print(f"⚠️ Unmapped domains (NaN organisation): {missing.sum()} rows")
print(missing)

⚠️ Unmapped domains (NaN organisation): 183 rows
domain
www.eventbrite.co.uk                                 23
                                                     10
meetoecd1.zoom.us                                    10
education.us18.list-manage.com                        8
lgiu.us3.list-manage.com                              6
                                                     ..
lancasteruk.estore.flywire.com                        1
earlyyearswales.hflip.co                              1
3a551fc8-7675-4cc5-9ecd-8697a47d348f.filesusr.com     1
www.canva.com                                         1
www.alcs.co.uk                                        1
Name: count, Length: 93, dtype: int64


# Assign categories to the organiastions 

In [35]:
org_to_category = {
    # ========================================
    # GOVERNMENT_PUBLIC_SECTOR
    # ========================================

    # government_legislature
    "all_party_parliamentary_group_schools":        ("government_public_sector", "government_legislature"),
    "council_of_europe":                            ("government_public_sector", "government_legislature"),
    "dods_monitoring":                              ("government_public_sector", "government_legislature"),
    "dutch_government":                             ("government_public_sector", "government_legislature"),
    "education_scotland":                           ("government_public_sector", "government_legislature"),
    "labour_party":                                 ("government_public_sector", "government_legislature"),
    "liberal_democrats":                            ("government_public_sector", "government_legislature"),
    "local_government_chronicle":                   ("government_public_sector", "government_legislature"),
    "local_government_information_unit":            ("government_public_sector", "government_legislature"),
    "national_audit_office":                        ("government_public_sector", "government_legislature"),
    "national_crime_agency":                        ("government_public_sector", "government_legislature"),
    "ni_government":                                ("government_public_sector", "government_legislature"),
    "office_for_national_statistics":               ("government_public_sector", "government_legislature"),
    "scottish_government":                          ("government_public_sector", "government_legislature"),
    "scottish_parliament":                          ("government_public_sector", "government_legislature"),
    "uk_civil_service_jobs":                        ("government_public_sector", "government_legislature"),
    "uk_government":                                ("government_public_sector", "government_legislature"),
    "uk_parliament":                                ("government_public_sector", "government_legislature"),
    "welsh_government":                             ("government_public_sector", "government_legislature"),
    "welsh_parliament":                             ("government_public_sector", "government_legislature"),

    # executive_non_departmental_public_body_ndpb
    "ofsted_blog":                                  ("government_public_sector", "executive_non_departmental_public_body_ndpb"),

    # international_organisation
    "oecd":                                         ("government_public_sector", "international_organisation"),
    "unesco":                                       ("government_public_sector", "international_organisation"),
    "unicef":                                       ("government_public_sector", "international_organisation"),

    # ========================================
    # ACADEMIC_SECTOR
    # ========================================

    # universities
    "harvard_graduate_school_of_education":         ("academic_sector", "universities"),
    "kings_college_london":                         ("academic_sector", "universities"),
    "london_school_of_economics":                   ("academic_sector", "universities"),
    "manchester_metropolitan_university":           ("academic_sector", "universities"),
    "nottingham_trent_university":                  ("academic_sector", "universities"),
    "queen_mary_university_london":                 ("academic_sector", "universities"),
    "ucl":                                          ("academic_sector", "universities"),
    "universitas_21":                               ("academic_sector", "universities"),
    "university_of_birmingham":                     ("academic_sector", "universities"),
    "university_of_bristol":                        ("academic_sector", "universities"),
    "university_of_dundee_surveys":                 ("academic_sector", "universities"),
    "university_of_east_anglia":                    ("academic_sector", "universities"),
    "university_of_nottingham":                     ("academic_sector", "universities"),
    "uwe_bristol_blog":                             ("academic_sector", "universities"),

    # academic_publisher_platform
    "bera_journals":                                ("academic_sector", "academic_publisher_platform"),
    "cambridge_university_press":                   ("academic_sector", "academic_publisher_platform"),
    "doi_via_ucl_proxy":                            ("academic_sector", "academic_publisher_platform"),
    "education_arxiv":                              ("academic_sector", "academic_publisher_platform"),
    "elsevier":                                     ("academic_sector", "academic_publisher_platform"),
    "elsevier_researcher_academy":                  ("academic_sector", "academic_publisher_platform"),
    "frontiers_journal":                            ("academic_sector", "academic_publisher_platform"),
    "jstor":                                        ("academic_sector", "academic_publisher_platform"),
    "jstor_daily":                                  ("academic_sector", "academic_publisher_platform"),
    "mdpi_journals":                                ("academic_sector", "academic_publisher_platform"),
    "repec_econpapers":                             ("academic_sector", "academic_publisher_platform"),
    "repec_ideas":                                  ("academic_sector", "academic_publisher_platform"),
    "researchgate":                                 ("academic_sector", "academic_publisher_platform"),
    "sage_journals":                                ("academic_sector", "academic_publisher_platform"),
    "sciencedirect":                                ("academic_sector", "academic_publisher_platform"),
    "taylor_and_francis":                           ("academic_sector", "academic_publisher_platform"),
    "wiley":                                        ("academic_sector", "academic_publisher_platform"),

    # academic_network
    "academy_of_social_sciences":                   ("academic_sector", "academic_network"),
    "ari_association_for_research_innovation":      ("academic_sector", "academic_network"),
    "bera":                                         ("academic_sector", "academic_network"),
    "british_academy":                              ("academic_sector", "academic_network"),
    "n8_research_partnership":                      ("academic_sector", "academic_network"),
    "society_for_research_into_higher_education":   ("academic_sector", "academic_network"),

    # ========================================
    # RESEARCH_EVIDENCE_SECTOR
    # ========================================

    # research_organisation
    "centre_for_economic_performance_lse":          ("research_evidence_sector", "research_organisation"),
    "echild_research_centre":                       ("research_evidence_sector", "research_organisation"),
    "national_education_policy_center":             ("research_evidence_sector", "research_organisation"),
    "nfer":                                         ("research_evidence_sector", "research_organisation"),
    "northern_health_science_alliance":             ("research_evidence_sector", "research_organisation"),
    "oii_edtech_equity":                            ("research_evidence_sector", "research_organisation"),
    "oxford_school_of_thought":                     ("research_evidence_sector", "research_organisation"),
    "research_improvement_for_policy_and_learning": ("research_evidence_sector", "research_organisation"),

    # research_institution
    "ada_lovelace_institute":                       ("research_evidence_sector", "research_institution"),
    "alan_turing_institute":                        ("research_evidence_sector", "research_institution"),
    "arc_institute":                                ("research_evidence_sector", "research_institution"),
    "kings_fund":                                   ("research_evidence_sector", "research_institution"),
    "research_on_research_institute":               ("research_evidence_sector", "research_institution"),
    "scottish_ai":                                  ("research_evidence_sector", "research_institution"),

    # research_project_initiative
    "coproduction_collective":                      ("research_evidence_sector", "research_project_initiative"),
    "working_lives_of_teachers":                    ("research_evidence_sector", "research_project_initiative"),

    # research_funder
    "leverhulme_trust":                             ("research_evidence_sector", "research_funder"),
    "nuffield":                                     ("research_evidence_sector", "research_funder"),
    "ukri":                                         ("research_evidence_sector", "research_funder"),

    # ========================================
    # CIVIL_SOCIETY_NONPROFIT_SECTOR
    # ========================================

    # labour_union
    "fda_union":                                    ("civil_society_nonprofit_sector", "labour_union"),
    "nasuwt_teachers_union":                        ("civil_society_nonprofit_sector", "labour_union"),
    "national_education_union":                     ("civil_society_nonprofit_sector", "labour_union"),

    # charity_ngo
    "5rights_foundation":                           ("civil_society_nonprofit_sector", "charity_ngo"),
    "action_for_children":                          ("civil_society_nonprofit_sector", "charity_ngo"),
    "barnardos":                                    ("civil_society_nonprofit_sector", "charity_ngo"),
    "centre_for_mental_health":                     ("civil_society_nonprofit_sector", "charity_ngo"),
    "centre_for_social_justice":                    ("civil_society_nonprofit_sector", "charity_ngo"),
    "centre_for_young_lives":                       ("civil_society_nonprofit_sector", "charity_ngo"),
    "child_poverty_action_group":                   ("civil_society_nonprofit_sector", "charity_ngo"),
    "children_in_wales":                            ("civil_society_nonprofit_sector", "charity_ngo"),
    "children_rights_alliance_england":             ("civil_society_nonprofit_sector", "charity_ngo"),
    "childrend_participation_in_schools":           ("civil_society_nonprofit_sector", "charity_ngo"),
    "childrens_commissioner":                       ("civil_society_nonprofit_sector", "charity_ngo"),
    "digital_poverty_alliance":                     ("civil_society_nonprofit_sector", "charity_ngo"),
    "education_support_charity":                    ("civil_society_nonprofit_sector", "charity_ngo"),
    "fairness_foundation":                          ("civil_society_nonprofit_sector", "charity_ngo"),
    "internet_matters":                             ("civil_society_nonprofit_sector", "charity_ngo"),
    "joseph_rowntree_foundation":                   ("civil_society_nonprofit_sector", "charity_ngo"),
    "magic_breakfast":                              ("civil_society_nonprofit_sector", "charity_ngo"),
    "national_literacy_trust":                      ("civil_society_nonprofit_sector", "charity_ngo"),
    "nspcc_learning":                               ("civil_society_nonprofit_sector", "charity_ngo"),
    "sutton_trust":                                 ("civil_society_nonprofit_sector", "charity_ngo"),
    "youth_endowment_fund":                         ("civil_society_nonprofit_sector", "charity_ngo"),

    # professional_network
    "ascl":                                         ("civil_society_nonprofit_sector", "professional_network"),
    "association_of_directors_of_childrens_services": ("civil_society_nonprofit_sector", "professional_network"),
    "charities_supporting_teachers_uk":             ("civil_society_nonprofit_sector", "professional_network"),
    "chartered_college_of_teaching":                ("civil_society_nonprofit_sector", "professional_network"),
    "chartered_institute_of_public_relations":      ("civil_society_nonprofit_sector", "professional_network"),
    "headmasters_and_headmistresses_conference":    ("civil_society_nonprofit_sector", "professional_network"),
    "national_association_head_teachers":           ("civil_society_nonprofit_sector", "professional_network"),

    # practitioner_organisation
    "ambition_institute":                           ("civil_society_nonprofit_sector", "practitioner_organisation"),
    "early_years_alliance":                         ("civil_society_nonprofit_sector", "practitioner_organisation"),
    "national_institute_of_teaching":               ("civil_society_nonprofit_sector", "practitioner_organisation"),
    "play_wales":                                   ("civil_society_nonprofit_sector", "practitioner_organisation"),
    "teach_first":                                  ("civil_society_nonprofit_sector", "practitioner_organisation"),

    # ========================================
    # KNOWLEDGE_MOBILISER_THINK_TANK_SECTOR
    # ========================================

    # think_tank
    "centre_for_education_and_youth":               ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "chandler_institute":                           ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "demos":                                        ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "edge_foundation":                              ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "edsk_think_tank":                              ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "epi":                                          ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "hepi":                                         ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "ifg":                                          ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "ifs":                                          ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "ippr":                                         ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "labour_together":                              ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "nesta":                                        ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "on_think_tanks":                               ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "tony_blair_institute":                         ("knowledge_mobiliser_think_tank_sector", "think_tank"),
    "wales_centre_for_public_policy":               ("knowledge_mobiliser_think_tank_sector", "think_tank"),

    # evidence_mobiliser
    "eef":                                          ("knowledge_mobiliser_think_tank_sector", "evidence_mobiliser"),
    "fft_ed_datalab":                               ("knowledge_mobiliser_think_tank_sector", "evidence_mobiliser"),
    "full_fact":                                    ("knowledge_mobiliser_think_tank_sector", "evidence_mobiliser"),
    "ippo":                                         ("knowledge_mobiliser_think_tank_sector", "evidence_mobiliser"),
    "teacher_tapp":                                 ("knowledge_mobiliser_think_tank_sector", "evidence_mobiliser"),
    "transforming_evidence":                        ("knowledge_mobiliser_think_tank_sector", "evidence_mobiliser"),

    # advocacy_organisation
    "cape_collaboration_for_public_engagement":     ("knowledge_mobiliser_think_tank_sector", "advocacy_organisation"),
    "education_appg":                               ("knowledge_mobiliser_think_tank_sector", "advocacy_organisation"),
    "options_2040_project":                         ("knowledge_mobiliser_think_tank_sector", "advocacy_organisation"),
    "shadow_panel_project":                         ("knowledge_mobiliser_think_tank_sector", "advocacy_organisation"),
    "teaching_commission":                          ("knowledge_mobiliser_think_tank_sector", "advocacy_organisation"),

    # ========================================
    # COMMERCIAL_PRIVATE_SECTOR
    # ========================================

    # consultancy
    "atkins_realis":                                ("commercial_private_sector", "consultancy"),
    "beyth_consultancy":                            ("commercial_private_sector", "consultancy"),
    "big_education":                                ("commercial_private_sector", "consultancy"),
    "oriel_square":                                 ("commercial_private_sector", "consultancy"),
    "staff_college":                                ("commercial_private_sector", "consultancy"),

    # edtech_education_business
    "ocr_exam_board":                               ("commercial_private_sector", "edtech_education_business"),
    "pearson":                                      ("commercial_private_sector", "edtech_education_business"),
    "twinkl":                                       ("commercial_private_sector", "edtech_education_business"),

    # industry_association
    "bett_show":                                    ("commercial_private_sector", "industry_association"),
    "digital_good_network":                         ("commercial_private_sector", "industry_association"),
    "edtech_innovation_hub":                        ("commercial_private_sector", "industry_association"),
    "edtech_strategy_lab":                          ("commercial_private_sector", "industry_association"),
    "tech_uk":                                      ("commercial_private_sector", "industry_association"),

    # ========================================
    # MEDIA_SECTOR
    # ========================================

    # news_media
    "bbc":                                          ("media_sector", "news_media"),
    "belfast_telegraph":                            ("media_sector", "news_media"),
    "daily_mirror":                                 ("media_sector", "news_media"),
    "daily_telegraph":                              ("media_sector", "news_media"),
    "evening_standard":                             ("media_sector", "news_media"),
    "financial_times":                              ("media_sector", "news_media"),
    "guardian":                                     ("media_sector", "news_media"),
    "hechinger_report":                             ("media_sector", "news_media"),
    "holyrood_magazine":                            ("media_sector", "news_media"),
    "independent":                                  ("media_sector", "news_media"),
    "inews":                                        ("media_sector", "news_media"),
    "labour_list":                                  ("media_sector", "news_media"),
    "nation_cymru":                                 ("media_sector", "news_media"),
    "politics_home":                                ("media_sector", "news_media"),
    "sky_news":                                     ("media_sector", "news_media"),
    "the_times":                                    ("media_sector", "news_media"),
    "times_literary_supplement":                    ("media_sector", "news_media"),
    "yorkshire_post":                               ("media_sector", "news_media"),

    # specialist_media
    "edtech_digest":                                ("media_sector", "specialist_media"),
    "fe_news":                                      ("media_sector", "specialist_media"),
    "fe_week":                                      ("media_sector", "specialist_media"),
    "fed":                                          ("media_sector", "specialist_media"),
    "nursery_world_magazine":                       ("media_sector", "specialist_media"),
    "schools_week":                                 ("media_sector", "specialist_media"),
    "techbullion":                                  ("media_sector", "specialist_media"),
    "tes":                                          ("media_sector", "specialist_media"),
    "wired_gov":                                    ("media_sector", "specialist_media"),
    "wonkhe":                                       ("media_sector", "specialist_media"),

    # commentary_platform
    "becky_allen_substack":                         ("media_sector", "commentary_platform"),
    "bennie_kara_substack":                         ("media_sector", "commentary_platform"),
    "conversation":                                 ("media_sector", "commentary_platform"),
    "magicsmoke_substack":                          ("media_sector", "commentary_platform"),
    "policy_manchester_blog":                       ("media_sector", "commentary_platform"),
    "rebecca_allen":                                ("media_sector", "commentary_platform"),
    "samf_substack":                                ("media_sector", "commentary_platform"),

    # ========================================
    # DIGITAL_SOCIAL_MEDIA_PLATFORMS
    # ========================================

    "linkedin":                                     ("digital_social_media_platforms", "social_media"),
    "twitter":                                      ("digital_social_media_platforms", "social_media"),

    # ========================================
    # OTHER_MISCELLANEOUS
    # ========================================

    "digital_youth_index":                          ("other_miscellaneous", "unclear"),
    "e_estonia":                                    ("other_miscellaneous", "government_initiative"),
    "funding_futures":                              ("other_miscellaneous", "unclear"),
    "gamayo":                                       ("other_miscellaneous", "unclear"),
    "inclusion_in_practice":                        ("other_miscellaneous", "unclear"),
    "innovate_ed":                                  ("other_miscellaneous", "unclear"),
    "inside_edge_training":                         ("other_miscellaneous", "unclear"),
    "issuu":                                        ("other_miscellaneous", "content_platform"),
    "local_ed_2025":                                ("other_miscellaneous", "unclear"),
    "london_design_biennale":                       ("other_miscellaneous", "cultural_organisation"),
    "lucas_education_foundation":                   ("other_miscellaneous", "unclear"),
    "new_visions_for_education":                    ("other_miscellaneous", "unclear"),
    "sustainable_school_leadership":                ("other_miscellaneous", "unclear"),
    "teacher_success":                              ("other_miscellaneous", "unclear"),
    "the_difference":                               ("other_miscellaneous", "unclear"),
    "tpea_association":                             ("other_miscellaneous", "unclear"),
    "transforming_society":                         ("other_miscellaneous", "unclear"),
    "upen":                                         ("other_miscellaneous", "unclear"),
    "upp_foundation":                               ("other_miscellaneous", "unclear"),
}

# Apply
mapped = df["organisation"].map(org_to_category)
df[["org_broad_category", "org_category"]] = mapped.apply(pd.Series)

# Sanity check — orgs with no category mapping
unmapped_orgs = df[df["org_broad_category"].isna()]["organisation"].value_counts()
print("⚠️ Orgs missing from org_to_category:\n", unmapped_orgs)

⚠️ Orgs missing from org_to_category:
 Series([], Name: count, dtype: int64)


In [36]:
# Turn the tuple series into two columns
df[["org_broad_category", "org_category"]] = mapped.apply(pd.Series)

In [37]:
df[["organisation", "org_broad_category", "org_category"]].head()

,organisation,org_broad_category,org_category
0,schools_week,media_sector,specialist_media
1,schools_week,media_sector,specialist_media
2,schools_week,media_sector,specialist_media
3,schools_week,media_sector,specialist_media
4,schools_week,media_sector,specialist_media


In [38]:
df["org_category"].value_counts(dropna=False)

org_category
government_legislature                         235
specialist_media                               206
NaN                                            183
universities                                   101
think_tank                                      89
news_media                                      85
academic_network                                57
charity_ngo                                     53
academic_publisher_platform                     48
evidence_mobiliser                              46
commentary_platform                             44
research_organisation                           42
unclear                                         35
research_funder                                 29
professional_network                            26
international_organisation                      24
research_institution                             9
labour_union                                     8
advocacy_organisation                            8
practitioner_organ

In [39]:
df["org_broad_category"].value_counts(dropna=False)

org_broad_category
media_sector                             335
government_public_sector                 260
academic_sector                          206
NaN                                      183
knowledge_mobiliser_think_tank_sector    143
civil_society_nonprofit_sector            94
research_evidence_sector                  83
other_miscellaneous                       38
commercial_private_sector                 14
digital_social_media_platforms             7
Name: count, dtype: int64

# Inspect "Title" and "Description" 


In [40]:
df[['title', 'description']].info()
df[['title', 'description']].isna().sum()
df[['title', 'description']].head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1363 entries, 0 to 1362
Data columns (total 2 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        1363 non-null   string
 1   description  1363 non-null   string
dtypes: string(2)
memory usage: 21.4 KB


,title,description
0,"Reject fewer teacher applicants, DfE tells trainers","Susan Acland-Hood, the DfE's permanent secretary, told providers a 7 per cent jump in applicants this year had not led to an equivalent rise in offers for courses."
1,Revealed: the experts advising ministers on teacher training reforms review,"The Department for Education has appointed an ""external steering group"" to review both the initial teacher training and early career frameworks, first launched in 2019. The group is made up of seven experts who are ""closely familiar"" with both reforms, as well as the ""underpinning evidence"". They will help ""shape the work of the review, scrutinising, supporting and challenging our thinking"", the DfE said."
2,Deadline 23 August 2023,"Education secretary Gillian Keegan has launched a call for evidence on using artificial intelligence (AI) like ChatGPT in schools ""to get the best"" out of the new technology."
3,Ofqual and DfE studying 'feasibility' of 'fully digital' exams,"Some exam boards are already piloting on-screen assessment, but research by AQA last year found teachers' biggest barrier to digital exams was a lack of infrastructure. https://schoolsweek.co.uk/ofqual-and-dfe-studying-feasibility-of-fully-digital-exams/"
4,Labour,"Revealed: The full details of Labour's education 'mission' Entitled 'Breaking down the barrier to opportunity', Labour will 'revise delivery' of the ECF and give more details on the plan to simplify the system of teacher incentives."
5,Digital Poverty Alliance,A charity whose vision is for everyone to access the life changing benefits that digital brings. Homepage - https://digitalpovertyalliance.org/
6,A long read from Nuffield funded research project ' Advancing Leadership Development in Early Years Education via Digitally Mediated Professional Learning',"In brief, the report finds:"
7,Post-16 Maths,"Nuffield Director of Education, John Hillman on 'The issues swirling round delivery of the PM's post-16 maths plan'"
8,Lib Dem,"Munira Wilson, Lib Dem spokesperson for education, is currently drawing up what she says is a ""strong education offer"" in the Lib Dem manifesto. Details seem thin on the ground so far. https://schoolsweek.co.uk/munira-wilson-lib-dem-education-spokesperson/"
9,Teacher retention commission: 8 proposals to stem exodus,"Teacher wellbeing chari ty Education Support has put forward a list of proposals to boost retention in the sector (published in partnership with Public First.) Report calls for review of teacher hours, retention targets and sabbaticals for headteachers every five years."


In [41]:
df['title_length'] = df['title'].str.len()
df['description_length'] = df['description'].str.len()
df[['title_length', 'description_length']].describe()

,title_length,description_length
count,1363.0,1363.0
mean,85.136464,230.955979
std,40.556777,365.241337
min,6.0,1.0
25%,61.0,114.0
50%,78.0,180.0
75%,101.0,275.0
max,482.0,11067.0


In [52]:
#Inspect titles with >50 words 
df['title_word_count'] = df['title'].apply(lambda x: len(str(x).split()))

long_titles = df[df['title_word_count'] > 30]
long_titles.head()




,id,newsletter_number,issue_date,theme,subtheme,title,description,link,new_theme,domain,organisation,org_broad_category,org_category,title_length,description_length,title_word_count,description_word_count,text,text_length_chars,text_length_words
70,ca997604-1f63-43b5-a74c-73192536ad52,6,8 September 2023,What Matters in Education?,<NA>,Why are Britain's kids refusing to go to school? As a new school year begins more children than ever were worried about returning. What's being done to get them into the classroom – and is that always the best idea?,https://www.theguardian.com/education/2023/sep/02/children-are-holding-a-mirror-up-to-us-why-are-britains-kids-refusing-to-go-to-school,https://www.theguardian.com/education/2023/sep/02/children-are-holding-a-mirror-up-to-us-why-are-britains-kids-refusing-to-go-to-school,what_matters_ed,www.theguardian.com,guardian,media_sector,news_media,215,135,40,1,Why are Britain's kids refusing to go to school? As a new school year begins more children than ever were worried about returning. What's being done to get them into the classroom – and is that always the best idea? https://www.theguardian.com/education/2023/sep/02/children-are-holding-a-mirror-up-to-us-why-are-britains-kids-refusing-to-go-to-school,351,41
72,904784d2-afc0-4210-96b4-536c54492695,6,8 September 2023,"Teacher recruitment, retention & development",<NA>,"Retaining high-quality teachers in disadvantaged communities is key to ensuring pupils' aspirations remain high, Teach First found, with new research showing that students from poorer backgrounds are more pessimistic about their future than their more well-off peers.",https://www.tes.com/magazine/news/secondary/retaining-teachers-schools-disadvantaged-pupils-aspirations,https://www.tes.com/magazine/news/secondary/retaining-teachers-schools-disadvantaged-pupils-aspirations,teacher_rrd,www.tes.com,tes,media_sector,specialist_media,267,103,37,1,"Retaining high-quality teachers in disadvantaged communities is key to ensuring pupils' aspirations remain high, Teach First found, with new research showing that students from poorer backgrounds are more pessimistic about their future than their more well-off peers. https://www.tes.com/magazine/news/secondary/retaining-teachers-schools-disadvantaged-pupils-aspirations",371,38
89,781404e3-e260-4c3c-9b82-c792e559b094,7,15 September 2023,Political landscape & key organisations,<NA>,"Report prepared for the Child of the North All-Party Parliamentary Group (APPG) highlights how a failure to address childhood inequality is creating a ""conveyor belt of disadvantage."" Also includes a chapter on how universities can work with LAs",APPG Report - https://www.thenhsa.co.uk/app/uploads/2023/09/APPG-REPORT-SEPT-23.pdf,https://www.thenhsa.co.uk/app/uploads/2023/09/APPG-REPORT-SEPT-23.pdf,political_environment_key_organisations,www.thenhsa.co.uk,northern_health_science_alliance,research_evidence_sector,research_organisation,245,83,38,4,"Report prepared for the Child of the North All-Party Parliamentary Group (APPG) highlights how a failure to address childhood inequality is creating a ""conveyor belt of disadvantage."" Also includes a chapter on how universities can work with LAs APPG Report - https://www.thenhsa.co.uk/app/uploads/2023/09/APPG-REPORT-SEPT-23.pdf",329,42
96,4e8f6f4c-2994-4ab2-9515-d206db4a41e4,8,19 October 2023,Conferences,<NA>,"Toby Greany presented papers on Leadership across Partnerships and Networks and was part of the panel discussion in a session organised by OECD on Talking the talk, walking the walk: Have we mobilised research on knowledge mobilisation?","Gemma Moss presented a paper called: Resetting Agendas for Educational Research Post COVID: Whose voice counts? Rob Klassen and Sophie Thompson-Lee attended EARLI in Thessaloniki and presented on 'Building a Better Understand of Teachers' Well-being.' Rob also chaired a panel on 'Revisting Effects of Teacher Characteristics on Stress: A Virtual Reality Study.' Sarah Chicken a

In [53]:
#Inspect descriptions with <10 words 

df['description_word_count'] = df['description'].apply(lambda x: len(str(x).split()))

short_descriptions = df[df['description_word_count'] < 10]

short_descriptions.head()


,id,newsletter_number,issue_date,theme,subtheme,title,description,link,new_theme,domain,organisation,org_broad_category,org_category,title_length,description_length,title_word_count,description_word_count,text,text_length_chars,text_length_words
6,de7e82ce-b30d-4d2a-ba49-c8823e2a2b53,1,11 July 2023,Thematic roundup,Digital,A long read from Nuffield funded research project ' Advancing Leadership Development in Early Years Education via Digitally Mediated Professional Learning',"In brief, the report finds:",https://www.nuffieldfoundation.org/wp-content/uploads/2021/10/Project-Report-Advancing-Ear-Years-Leadership-Development.pdf,edtech,www.nuffieldfoundation.org,nuffield,research_evidence_sector,research_funder,155,27,21,5,"A long read from Nuffield funded research project ' Advancing Leadership Development in Early Years Education via Digitally Mediated Professional Learning' In brief, the report finds:",183,26
11,e7d3d53b-19c8-4c30-9c68-0de5e15f5ea3,2,16 July 2023,Events,<NA>,Date: Tuesday 18 th July 11:30 to 13:30 (CET),Zoom Registration Link here.,https://meetoecd1.zoom.us/meeting/register/tJwkdemopj0pHNEksLPioTqAj_qyuIz3VMkH,update_from_programme,meetoecd1.zoom.us,NaN,NaN,NaN,45,28,9,4,Date: Tuesday 18 th July 11:30 to 13:30 (CET) Zoom Registration Link here.,74,13
12,54a96f40-2990-4b7a-9a30-695bf9e911a1,2,16 July 2023,PI Updates and Papers,Leadership,A three-part series on leadership from Toby Greany and team on TES:,Part 1: Headteacher recruitment crisis: 5 tips for action,https://www.tes.com/magazine/leadership/staff-management/headteacher-recruitment-crisis-applications,update_from_pi,www.tes.com,tes,media_sector,specialist_media,67,57,12,9,A three-part series on leadership from Toby Greany and team on TES: Part 1: Headteacher recruitment crisis: 5 tips for action,125,21
18,5b973a30-00ca-4e78-b8fb-0b110b1f352a,2,16 July 2023,PI Updates and Papers,Digital,Cutting through the conjecture: How is EdTech really being used in our classrooms?,Full post - https://edtech.oii.ox.ac.uk/cutting-through-the-conjecture/,https://edtech.oii.ox.ac.uk/cutting-through-the-conjecture,edtech,edtech.oii.ox.ac.uk,oii_edtech_equity,research_evidence_sector,research_organisation,82,71,13,4,Cutting through the conjecture: How is EdTech really being used in our classrooms? Full post - https://edtech.oii.ox.ac.uk/cutting-through-the-conjecture/,154,17
35,431837f7-b09d-429e-97ce-452f56f732cb,3,20 July 2023,PI Updates and Papers,<NA>,This research paper brings together three of the young journalists who worked on The Corona Times Journal to reflect on their experiences of being involved in the project.,https://www.tandfonline.com/doi/epdf/10.1080/13642987.2022.2061954?needAccess=true&role=button,https://www.tandfonline.com/doi/epdf/10.1080/13642987.2022.2061954?needAccess=true&role=button,update_from_pi,www.tandfonline.com,taylor_and_francis,academic_sector,academic_publisher_platform,171,94,28,1,This research paper brings together three of the young journalists who worked on The Corona Times Journal to reflect on their experiences of being involved in the project. https://www.tandfonline.com/doi/epdf/10.1080/13642987.2022.2061954?needAccess=true&role=button,266,29


#### Create 'Text' Variable = 'Title' + 'Description'

In [54]:
#Create "Text" variable = "Title" + "Description" 
df['text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')

In [55]:
# Basic info on the new column
print(df['text'].info())

# Add a column for text length (number of words or characters)
df['text_length_chars'] = df['text'].str.len()
df['text_length_words'] = df['text'].str.split().str.len()

# Summary statistics
print("\nCharacter length stats:")
print(df['text_length_chars'].describe())

<class 'pandas.core.series.Series'>
RangeIndex: 1363 entries, 0 to 1362
Series name: text
Non-Null Count  Dtype 
--------------  ----- 
1363 non-null   string
dtypes: string(1)
memory usage: 10.8 KB
None

Character length stats:
count        1363.0
mean     317.092443
std      365.355728
min            52.0
25%           193.5
50%           267.0
75%           363.0
max         11156.0
Name: text_length_chars, dtype: Float64


In [56]:
# Check for missing or empty values
missing_mask = df['text'].isna() | (df['text'].str.strip() == '')

# Count how many
missing_count = missing_mask.sum()
print(f"Missing or empty 'text' entries: {missing_count}")

# Optionally view them
if missing_count > 0:
    print(df.loc[missing_mask, ['title', 'description']].head())


Missing or empty 'text' entries: 0


In [57]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1363 entries, 0 to 1362
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id                      1363 non-null   string
 1   newsletter_number       1363 non-null   int64 
 2   issue_date              1363 non-null   string
 3   theme                   1363 non-null   string
 4   subtheme                80 non-null     string
 5   title                   1363 non-null   string
 6   description             1363 non-null   string
 7   link                    1363 non-null   string
 8   new_theme               1363 non-null   object
 9   domain                  1363 non-null   object
 10  organisation            1180 non-null   object
 11  org_broad_category      1180 non-null   object
 12  org_category            1180 non-null   object
 13  title_length            1363 non-null   Int64 
 14  description_length      1363 non-null   Int64 
 15  titl

In [59]:
df.columns

Index(['id', 'newsletter_number', 'issue_date', 'theme', 'subtheme', 'title',
       'description', 'link', 'new_theme', 'domain', 'organisation',
       'org_broad_category', 'org_category', 'title_length',
       'description_length', 'title_word_count', 'description_word_count',
       'text', 'text_length_chars', 'text_length_words'],
      dtype='object')

# Save Files 

In [60]:
df.to_csv("/workspaces/AM2_erp_programme_automataion/data/preprocessed/newsletters_clean.csv", index=False)

print(f"✅ ✅ ✅ Saved")

✅ ✅ ✅ Saved


In [ ]:
df.head(0)